# LoRA Fine-Tuning for Fall Detection

This notebook fine-tunes a language model using LoRA (Low-Rank Adaptation) for fall detection classification.

**Requirements:**
- Google Colab with GPU (T4 free tier works)
- Upload your training data

**Model:** Mistral-7B-Instruct (or smaller alternative)

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install -q huggingface_hub

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import json
import os

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload Training Data

Upload your `train.jsonl` file from `final_pipeline/lora_training/`

In [ ]:
from google.colab import files

# Upload your training data
print("Upload train.jsonl from final_pipeline/lora_training/")
uploaded = files.upload()

In [ ]:
# Load and prepare training data
def load_training_data(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            item = json.loads(line)
            messages = item['messages']
            
            # Convert to instruction format
            system = next((m['content'] for m in messages if m['role'] == 'system'), '')
            user = next((m['content'] for m in messages if m['role'] == 'user'), '')
            assistant = next((m['content'] for m in messages if m['role'] == 'assistant'), '')
            
            text = f"""### Instruction:
{system}

### Input:
{user}

### Response:
{assistant}"""
            data.append({'text': text})
    return data

# Load data
train_data = load_training_data('train.jsonl')
print(f"Loaded {len(train_data)} training examples")
print(f"\nSample:\n{train_data[0]['text'][:500]}...")

In [ ]:
# Create HuggingFace dataset
dataset = Dataset.from_list(train_data)
dataset = dataset.shuffle(seed=42)

# Split into train/val
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
val_dataset = split['test']

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## 3. Load Base Model with Quantization

In [ ]:
# Model configuration
# Using a smaller model that fits in Colab's free GPU
MODEL_NAME = "microsoft/phi-2"  # 2.7B params, fits in T4
# Alternative: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model: {MODEL_NAME}")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare for LoRA training
model = prepare_model_for_kbit_training(model)

print("Model loaded successfully!")

## 4. Configure LoRA

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=16,                      # LoRA rank
    lora_alpha=32,             # Alpha scaling
    lora_dropout=0.05,         # Dropout
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "dense"],  # Layers to adapt
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## 5. Training

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./lora_fall_detection",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit",
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512,
    packing=False,
)

print("Trainer ready!")

In [ ]:
# Start training
print("Starting LoRA fine-tuning...")
trainer.train()
print("Training complete!")

In [ ]:
# Save the LoRA adapter
trainer.save_model("./lora_fall_detection_final")
print("Model saved!")

## 6. Evaluation

In [ ]:
# Test the fine-tuned model
def predict(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=0.1,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test example
test_prompt = """### Instruction:
You are a fall detection classifier. Analyze the pose data and respond with FALL or NO_FALL.

### Input:
Frame 1: hip_y=0.45, angle=85, velocity=0.01, posture=upright
Frame 2: hip_y=0.72, angle=35, velocity=0.15, posture=fallen, FLAGS: ON_GROUND, HORIZONTAL
Frame 3: hip_y=0.78, angle=25, velocity=0.02, posture=fallen

### Response:
"""

result = predict(test_prompt, model, tokenizer)
print(result)

In [ ]:
# Evaluate on validation set
from tqdm import tqdm

correct = 0
total = 0
fall_correct = 0
fall_total = 0

for item in tqdm(val_dataset):
    text = item['text']
    # Extract true label from the text
    true_label = "FALL" if "FALL" in text.split("### Response:")[-1] and "NO_FALL" not in text.split("### Response:")[-1] else "NO_FALL"
    
    # Get prediction
    prompt = text.split("### Response:")[0] + "### Response:\n"
    pred = predict(prompt, model, tokenizer)
    pred_label = "FALL" if "FALL" in pred.split("### Response:")[-1] and "NO_FALL" not in pred.split("### Response:")[-1] else "NO_FALL"
    
    if pred_label == true_label:
        correct += 1
    total += 1
    
    if true_label == "FALL":
        fall_total += 1
        if pred_label == "FALL":
            fall_correct += 1

print(f"\n=== LoRA Model Results ===")
print(f"Accuracy: {correct/total*100:.1f}%")
print(f"Fall Recall: {fall_correct/fall_total*100:.1f}%" if fall_total > 0 else "No fall samples")

## 7. Download Model

In [ ]:
# Zip and download the LoRA adapter
!zip -r lora_fall_detection.zip ./lora_fall_detection_final

from google.colab import files
files.download('lora_fall_detection.zip')

## Summary

This notebook demonstrates:
1. **LoRA Fine-Tuning** using PEFT library
2. **4-bit Quantization** for memory efficiency
3. **Training** on fall detection data
4. **Evaluation** with accuracy and recall metrics

The LoRA adapter can be saved and used for inference.